# 02. 전처리

음식 데이터 정제와 전처리 방법을 실험한다.

- 결측치·중복 처리 방식 실험
- 음식명·설명 텍스트 정규화
- 추천에 사용할 컬럼 선별 및 정리
- 검증된 로직은 `src/preprocessing/`으로 분리

입력: `data/raw/` / 출력: `data/processed/`

01 분석 결과를 바탕으로 다음 순서로 진행한다.

1. 메뉴 그룹 분류: 1차 추천 대상(식사)과 반찬, 디저트, 음료 구분
2. 식품명 정제: 원본 보존, 추천용 메뉴명과 온도, 사이즈 속성 분리
3. 중복 처리: 업체, 출처, 영양성분을 고려한 통합 기준
4. 컬럼 선별과 `해당없음` 처리
5. 식품중량 분리
6. `data/processed/` 저장과 전후 비교

각 단계의 검증된 로직은 `src/preprocessing/food_data.py`에 있으며 이 노트북은 그 함수를 실제 데이터에 적용하며 근거를 기록한다.

## 기본 설정

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_PATH = DATA_DIR / "raw" / "food_nutrition.csv"
PROCESSED_DIR = DATA_DIR / "processed"
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import food_data as fd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

In [2]:
raw = fd.load_raw(RAW_PATH)
raw.shape

(19617, 50)

## 1. 메뉴 그룹 분류

01 분석에서 빵 및 과자류와 음료 및 차류가 전체의 약 73%였다. 1차 추천 대상을 식사 메뉴로 잡기 위해 `식품대분류명`을 기준으로 다섯 그룹으로 나눈다.

| 그룹 | 기준 |
|---|---|
| 식사 | 밥, 면·만두, 국·탕, 찌개·전골, 죽·스프, 볶음, 구이, 튀김, 찜, 조림, 전·적·부침 |
| 반찬 | 생채·무침, 나물·숙채, 김치, 장아찌·절임, 젓갈, 장류·양념 |
| 디저트 | 빵 및 과자류, 유제품류 및 빙과류 |
| 음료 | 음료 및 차류 |
| 기타 | 수·조·어·육류, 곡류·서류 제품, 채소·해조류, 과일류, 두류·견과 (원재료 성격, 17행) |

빵 및 과자류 안에는 피자, 버거, 샌드위치처럼 한 끼 식사로 먹는 메뉴가 섞여 있다. 어떤 대표식품명이 있는지 확인한다.

In [3]:
raw.loc[raw["식품대분류명"] == "빵 및 과자류", "대표식품명"].value_counts().head(20)

대표식품명
피자            4710
케이크            657
버거             295
도넛             285
샌드위치           233
와플             199
마카롱            187
크로플            162
베이글            136
햄버거            133
크림빵            113
비스킷/쿠키/크래커     106
식빵             105
크로와상            96
페이스트리           79
치즈빵             66
크로켓(고로케)        62
핫도그             61
머핀              60
프레즐             57
Name: count, dtype: int64

피자, 버거, 햄버거, 샌드위치, 핫도그, 토스트는 식사로 분류한다. 케이크, 도넛, 와플 등은 디저트로 둔다.

반찬은 단독 메뉴로 추천하기 어렵고 급식 데이터 성격이 강해 1차 대상에서 제외하되, 그룹 라벨을 남겨 나중에 활용할 수 있게 한다.

In [4]:
work = raw.copy()
work["프랜차이즈여부"] = work["업체명"] != fd.NOT_APPLICABLE
work["메뉴그룹"] = fd.assign_menu_group(work)

group_counts = (
    work.groupby(["메뉴그룹", "식품대분류명"]).size().rename("행 수").reset_index()
    .sort_values(["메뉴그룹", "행 수"], ascending=[True, False])
)
group_counts

,메뉴그룹,식품대분류명,행 수
3,기타,수·조·어·육류,8
0,기타,"곡류, 서류 제품",6
1,기타,과일류,1
2,기타,"두류, 견과 및 종실류",1
4,기타,"채소, 해조류",1
5,디저트,빵 및 과자류,3177
6,디저트,유제품류 및 빙과류,663
9,반찬,생채·무침류,501
8,반찬,나물·숙채류,248
7,반찬,김치류,52


In [5]:
pd.crosstab(work["메뉴그룹"], work["프랜차이즈여부"], margins=True)

프랜차이즈여부,False,True,All
메뉴그룹,,,
기타,14,3,17
디저트,116,3724,3840
반찬,821,46,867
식사,3139,5978,9117
음료,54,5722,5776
All,4144,15473,19617


식사 그룹 안에서도 프랜차이즈 피자가 큰 비중을 차지한다. 프랜차이즈 여부를 컬럼으로 남겨 추천 단계에서 필터링할 수 있게 한다.

In [6]:
meal = work[work["메뉴그룹"] == fd.MENU_GROUP_MEAL]
meal.groupby(["식품대분류명", "프랜차이즈여부"]).size().unstack(fill_value=0).sort_values(True, ascending=False)

프랜차이즈여부,False,True
식품대분류명,,
빵 및 과자류,37,5429
튀김류,218,308
면 및 만두류,274,70
볶음류,459,47
찌개 및 전골류,301,39
구이류,260,29
밥류,398,26
국 및 탕류,503,10
죽 및 스프류,95,10


## 2. 식품명 정제

원본 `식품명`은 그대로 두고 추천용 `메뉴명`을 추가한다. 01 분석에서 확인한 표기 형태에 따라 규칙을 다르게 적용한다.

- 프랜차이즈 (`업체명`이 있는 행): `카테고리_메뉴명 온도 (사이즈)` 형태. 첫 `_` 앞은 `이름접두어`로 분리
- 비프랜차이즈: `_`는 재료 변형 구분자 (예: `된장국_근대`). 공백으로 치환
- `핫(HOT)`, `아이스(ICED)`는 `온도` 컬럼으로 분리
- 괄호 안 값이 사이즈 표기면 `사이즈` 컬럼으로 분리. 그 외 괄호 (조각, 8개입 등)는 이름에 유지

먼저 괄호 안에 어떤 값이 있는지 확인해 사이즈 표기를 결정한다.

In [7]:
paren_tokens = raw["식품명"].str.findall(r"\(([^()]*)\)").explode().dropna()
paren_tokens.value_counts().head(40)

식품명
L             3049
ICED          1890
R             1485
HOT           1449
M              590
F              227
Tall           179
P              153
XL             145
Venti          142
ML             133
Grande         128
EX             103
Mini Venti      94
J               91
S               68
조각              62
고로케             62
더벤티             52
1인              32
V               31
Max             29
G               28
코끼리             28
치즈              26
소               23
액상              23
H               18
대               16
8개입             15
초대용량            15
5개입             14
홀               14
닭갈비             14
3개입             13
20개입            10
2개입             10
싱글              10
더블              10
1개입              8
Name: count, dtype: int64

L, R, M, S, XL, Tall, Grande, Venti 등 음료·피자 사이즈 표기만 사이즈로 인정한다. `조각`, `N개입`, `1인`처럼 수량 정보는 이름에 남긴다.

`아이스`, `핫` 단독 표기는 `아이스크림`과 겹치므로 `(HOT)`, `(ICED)` 토큰이 있는 경우만 온도로 추출한다.

In [8]:
print("사이즈로 인정하는 표기:", sorted(fd.SIZE_TOKENS))

사이즈로 인정하는 표기: ['EX', 'F', 'G', 'Grande', 'H', 'J', 'L', 'M', 'ML', 'Max', 'Mini Venti', 'P', 'R', 'S', 'Short', 'Solo', 'Tall', 'V', 'Venti', 'XL', '대', '더벤티', '소', '중', '초대용량']


In [9]:
work = fd.add_name_columns(work)

work.loc[work["프랜차이즈여부"], ["식품명", "메뉴명", "이름접두어", "온도", "사이즈"]].sample(12, random_state=0)

,식품명,메뉴명,이름접두어,온도,사이즈
3994,피자_버팔로 피자 치즈크러스트 (R),버팔로 피자 치즈크러스트,피자,NaN,R
1854,피자_크리스피 치즈 페퍼로니,크리스피 치즈 페퍼로니,피자,NaN,NaN
7990,커피_아메리카노 핫(HOT),아메리카노,커피,HOT,NaN
10914,아이스크림_하루한번하늘 그릭요거트,하루한번하늘 그릭요거트,아이스크림,NaN,NaN
1418,피자_페페로니 피자 씬도우 (L),페페로니 피자 씬도우,피자,NaN,L
2537,피자_전주불백피자 (XL),전주불백피자,피자,NaN,XL
14859,밀크티/버블티_타로 버블티,타로 버블티,밀크티/버블티,NaN,NaN
15846,레몬차_레몬티 핫(HOT),레몬티,레몬차,HOT,NaN
6633,케이크_오리지널 티라미수 케이크,오리지널 티라미수 케이크,케이크,NaN,NaN
6039,크로플_마약크림 딸기 크로플,마약크림 딸기 크로플,크로플,NaN,NaN


In [10]:
work.loc[~work["프랜차이즈여부"], ["식품명", "메뉴명", "이름접두어", "온도", "사이즈"]].sample(8, random_state=0)

,식품명,메뉴명,이름접두어,온도,사이즈
9170,청포묵 무침,청포묵 무침,NaN,NaN,NaN
9122,취나물무침,취나물무침,NaN,NaN,NaN
15563,막국수,막국수,NaN,NaN,NaN
9334,죽순볶음,죽순볶음,NaN,NaN,NaN
13559,삼계탕,삼계탕,NaN,NaN,NaN
12929,소시지볶음,소시지볶음,NaN,NaN,NaN
18194,다시마튀각,다시마튀각,NaN,NaN,NaN
18178,달걀 샐러드,달걀 샐러드,NaN,NaN,NaN


In [11]:
print("온도 분포"); print(work["온도"].value_counts(dropna=False))
print()
print("사이즈 분포 (상위)"); print(work["사이즈"].value_counts(dropna=False).head(12))

온도 분포
온도
NaN     16278
ICED     1890
HOT      1449
Name: count, dtype: int64

사이즈 분포 (상위)
사이즈
NaN       12806
L          3049
R          1485
M           590
F           227
Tall        179
P           153
XL          145
Venti       142
ML          133
Grande      128
EX          103
Name: count, dtype: int64


In [12]:
# 정제 후 메뉴명에 남은 괄호 표기 확인 (사이즈로 처리되지 않은 값)
work["메뉴명"].str.findall(r"\(([^()]*)\)").explode().dropna().value_counts().head(15)

메뉴명
조각      62
1인      32
코끼리     28
치즈      26
8개입     15
5개입     14
홀       14
3개입     13
닭갈비     11
20개입    10
2개입     10
싱글      10
더블      10
1개입      8
6개입      7
Name: count, dtype: int64

In [13]:
# 정제 후 메뉴명 고유값 변화
print(f"식품명 고유값: {work['식품명'].nunique():,}")
print(f"메뉴명 고유값: {work['메뉴명'].nunique():,}")
print(f"메뉴명 + 업체명 고유값: {work[['메뉴명', '업체명']].drop_duplicates().shape[0]:,}")

식품명 고유값: 15,647
메뉴명 고유값: 11,454
메뉴명 + 업체명 고유값: 13,096


## 3. 식품중량 분리

`식품중량`은 `291.90ml`처럼 숫자와 단위가 붙은 문자열이다. 숫자 `중량값`과 단위 `중량단위`(g, ml)로 분리만 하고 환산이나 1인분 해석은 하지 않는다. `1인(회)분량 참고량`이 전부 결측이라 `식품중량`이 1인분인지 확인할 근거가 없다.

In [14]:
work = pd.concat([work, fd.parse_weight(work["식품중량"])], axis=1)

parse_failed = work["식품중량"].notna() & work["중량값"].isna()
print(f"원본 결측: {work['식품중량'].isna().sum()}")
print(f"형식 불일치로 분리 실패: {parse_failed.sum()}")
print()
print(work["중량단위"].value_counts(dropna=False))

원본 결측: 52
형식 불일치로 분리 실패: 0

중량단위
g      13825
ml      5740
NaN       52
Name: count, dtype: int64


In [15]:
work[["식품명", "영양성분함량기준량", "식품중량", "중량값", "중량단위"]].sample(6, random_state=1)

,식품명,영양성분함량기준량,식품중량,중량값,중량단위
4765,피자_닭발 피자 (L),100g,932g,932.0,g
5794,타르트_스모어마시멜로우타르트,100g,50g,50.0,g
4641,피자_도이치 피자 밀도우 (L),100g,1120g,1120.0,g
942,피자_할루미체다크림크러스트(L),100g,1020g,1020.0,g
12068,스무디_제주 그린티 스무디,100g,473g,473.0,g
2320,피자_치왕체다크림크러스트,100g,1305g,1305.0,g


In [16]:
# 기준량 단위와 중량 단위가 다른 경우 확인
pd.crosstab(work["영양성분함량기준량"], work["중량단위"].fillna("결측"))

중량단위,g,ml,결측
영양성분함량기준량,,,
100g,13825,0,52
100ml,0,5740,0


## 4. 중복 처리 기준

01 분석에서 식품명이 같은 행이 3,970행 있었지만 원인이 두 가지였다.

- 급식 데이터: 같은 음식이 초등·중고등·산업체 급식으로 반복되며 영양성분도 동일
- 프랜차이즈: 같은 메뉴명이 여러 업체에 있으며 영양성분은 다름

음식명만으로 제거하면 업체별로 다른 메뉴가 사라진다. 따라서 다음 컬럼이 모두 같은 행만 하나로 통합한다.

- `식품명`, `업체명`, `영양성분함량기준량`, `식품중량`
- 영양성분 25개 컬럼 전체 (결측은 결측끼리 같은 것으로 간주)

통합 시 `식품코드` 오름차순 첫 행을 남기고, 제거된 행은 `대표식품코드`에 매핑해 별도 파일로 저장한다.

In [17]:
before_dedup = len(work)
work, dedup_map = fd.deduplicate(work)

print(f"통합 전: {before_dedup:,}행")
print(f"통합 후: {len(work):,}행")
print(f"제거 (매핑표 기록): {len(dedup_map):,}행")

통합 전: 19,617행
통합 후: 18,998행
제거 (매핑표 기록): 619행


In [18]:
dedup_map.head(10)

,식품코드,대표식품코드,식품명,식품기원명,업체명
0,D501-003000000-0001,D401-003000000-0001,곤드레밥,초등학교급식(재료량 기반 산출 함량),해당없음
1,D501-007480000-0001,D401-007480000-0001,김밥_채소,초등학교급식(재료량 기반 산출 함량),해당없음
2,D501-017030000-0001,D401-017030000-0001,볶음밥_계란,초등학교급식(재료량 기반 산출 함량),해당없음
3,D501-017480000-0001,D401-017480000-0001,볶음밥_채소,초등학교급식(재료량 기반 산출 함량),해당없음
4,D501-032600000-0001,D401-032600000-0001,잡곡밥_보리,초등학교급식(재료량 기반 산출 함량),해당없음
5,D501-035000000-0001,D401-035000000-0001,주먹밥,초등학교급식(재료량 기반 산출 함량),해당없음
6,D501-068000000-0001,D401-068000000-0001,버섯 덮밥,초등학교급식(재료량 기반 산출 함량),해당없음
7,D501-073000000-0001,D401-073000000-0001,양송이 덮밥,초등학교급식(재료량 기반 산출 함량),해당없음
8,D501-077000000-0001,D401-077000000-0001,완두콩밥,초등학교급식(재료량 기반 산출 함량),해당없음
9,D502-077000000-0001,D402-077000000-0001,계란빵,초등학교급식(재료량 기반 산출 함량),해당없음


In [19]:
# 제거된 행의 출처: 급식 반복 데이터만 해당하는지 확인
dedup_map["식품기원명"].value_counts()

식품기원명
산업체급식(재료량 기반 산출 함량)     295
중고등학교급식(재료량 기반 산출함량)    234
초등학교급식(재료량 기반 산출 함량)     90
Name: count, dtype: int64

In [20]:
# 통합 예시: 흰죽
sample_code = dedup_map.loc[dedup_map["식품명"] == "흰죽", "대표식품코드"].iloc[0]
print("대표 행")
display(work.loc[work["식품코드"] == sample_code, ["식품코드", "식품명", "식품기원명", "에너지(kcal)", "나트륨(mg)"]])
print("매핑된 행")
display(dedup_map[dedup_map["대표식품코드"] == sample_code])

대표 행


,식품코드,식품명,식품기원명,에너지(kcal),나트륨(mg)
0,D504-212000000-0001,흰죽,초등학교급식(재료량 기반 산출 함량),64,130.0


매핑된 행


,식품코드,대표식품코드,식품명,식품기원명,업체명
134,D604-212000000-0001,D504-212000000-0001,흰죽,중고등학교급식(재료량 기반 산출함량),해당없음
362,D704-212000000-0001,D504-212000000-0001,흰죽,산업체급식(재료량 기반 산출 함량),해당없음


In [21]:
# 식품명만 같고 통합되지 않은 예시: 업체별 영양성분이 다름
work.loc[work["식품명"] == "커피_아메리카노 핫(HOT)", ["식품코드", "업체명", "식품중량", "에너지(kcal)", "당류(g)", "나트륨(mg)"]].head(8)

,식품코드,업체명,식품중량,에너지(kcal),당류(g),나트륨(mg)
7982,D220-748080000-0003,컴포즈커피,591ml,3,0.0,1.0
7983,D220-748080000-0069,할리스,354ml,3,0.0,1.0
7984,D220-748080000-0076,더벤티,600ml,2,NaN,NaN
7985,D220-748000000-0945,베러댄와플,340g,3,0.0,1.0
7986,D220-748000000-0943,매머드 익스프레스,473g,2,NaN,0.0
7987,D220-748000000-0941,달콤,355g,2,0.0,0.0
7988,D220-748080000-0079,커피베이,360ml,1,0.0,0.0
7989,D220-748080000-0080,빽다방,473ml,3,0.0,1.0


## 5. 컬럼 선별과 `해당없음` 처리

01 분석에서 정리한 역할에 따라 추천에 필요한 컬럼만 남긴다.

| 역할 | 유지 컬럼 |
|---|---|
| 식별자 | `식품코드` |
| 음식명 | `식품명`(원본), `메뉴명`, `이름접두어`, `온도`, `사이즈` |
| 분류 | `메뉴그룹`, `식품대분류명`, `대표식품명`, `식품중분류명` |
| 출처 | `식품기원명`, `업체명`, `프랜차이즈여부`, `출처명`, `데이터생성방법명` |
| 제공량 | `영양성분함량기준량`, `중량값`, `중량단위` |
| 영양성분 | 에너지, 단백질, 지방, 탄수화물, 당류, 식이섬유, 나트륨, 콜레스테롤, 포화지방산 |

제외: 전체 결측인 `1인(회)분량 참고량`, 고유값 1개인 컬럼, 명칭과 1:1 대응하는 코드 컬럼, 관리용 날짜, `해당없음` 비율이 높은 소분류·세분류, 결측 68% 이상인 미량영양소.

영양성분 결측은 그대로 둔다. 프랜차이즈 데이터의 미량영양소 결측은 값이 0이 아니라 측정하지 않은 것이다.

`해당없음`은 컬럼 의미에 따라 다르게 처리한다.

- `업체명`: 업체가 없다는 뜻이므로 결측(NaN)으로 바꾸고 `프랜차이즈여부`로 구분
- `식품중분류명`: 분류가 없다는 뜻이므로 결측으로 바꿈
- `식품소분류명`, `식품세분류명`: 대부분 `해당없음`이라 컬럼 자체를 제외

In [22]:
work = fd.replace_not_applicable(work, ["업체명", "식품중분류명"])
clean = work[fd.OUTPUT_COLUMNS].reset_index(drop=True)

print(f"컬럼 수: {raw.shape[1]} -> {clean.shape[1]}")
clean.head(3).T

컬럼 수: 50 -> 27


,0,1,2
식품코드,D504-212000000-0001,D104-198000000-0001,D404-198000000-0001
식품명,흰죽,흑임자죽,흑임자 죽
메뉴명,흰죽,흑임자죽,흑임자 죽
이름접두어,NaN,NaN,NaN
온도,NaN,NaN,NaN
사이즈,NaN,NaN,NaN
메뉴그룹,식사,식사,식사
식품대분류명,죽 및 스프류,죽 및 스프류,죽 및 스프류
대표식품명,흰죽,흑임자죽,흑임자 죽
식품중분류명,NaN,NaN,NaN


In [23]:
# 영양성분 결측은 유지되었는지 확인 (0으로 채우지 않음)
pd.DataFrame({
    "결측 수": clean[fd.NUTRITION_COLUMNS_SELECTED].isna().sum(),
    "결측 비율(%)": (clean[fd.NUTRITION_COLUMNS_SELECTED].isna().mean() * 100).round(1),
    "0인 행 수": (clean[fd.NUTRITION_COLUMNS_SELECTED] == 0).sum(),
})

,결측 수,결측 비율(%),0인 행 수
에너지(kcal),0,0.0,160
단백질(g),0,0.0,672
지방(g),13319,70.1,137
탄수화물(g),12777,67.3,44
당류(g),149,0.8,755
식이섬유(g),15278,80.4,108
나트륨(mg),61,0.3,581
콜레스테롤(mg),13486,71.0,1256
포화지방산(g),592,3.1,1460


## 6. 저장 및 전후 비교

전체 파이프라인을 `fd.preprocess`로 한 번에 실행해 위 단계별 결과와 동일한지 확인한 뒤 저장한다.

- `food_clean.csv`: 전체 그룹 정제 결과
- `food_menu.csv`: 1차 추천 대상 (메뉴그룹 = 식사)
- `dedup_map.csv`: 통합된 행의 원본 식품코드와 대표 식품코드

In [24]:
clean_pipeline, dedup_map_pipeline = fd.preprocess(raw)

pd.testing.assert_frame_equal(clean, clean_pipeline)
pd.testing.assert_frame_equal(dedup_map, dedup_map_pipeline)
print("단계별 결과와 파이프라인 결과 일치")

단계별 결과와 파이프라인 결과 일치

In [25]:
menu = fd.select_meal_menu(clean)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED_DIR / "food_clean.csv", index=False, encoding="utf-8-sig")
menu.to_csv(PROCESSED_DIR / "food_menu.csv", index=False, encoding="utf-8-sig")
dedup_map.to_csv(PROCESSED_DIR / "dedup_map.csv", index=False, encoding="utf-8-sig")

for f in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"{f.name}: {f.stat().st_size / 1024:.0f} KB")

dedup_map.csv: 73 KB
food_clean.csv: 5134 KB
food_menu.csv: 2299 KB


### 행 수 비교

In [26]:
pd.DataFrame({
    "행 수": {
        "원본": len(raw),
        "중복 통합 후 (food_clean)": len(clean),
        "1차 추천 대상 (food_menu)": len(menu),
        "통합 매핑 (dedup_map)": len(dedup_map),
    }
})

,행 수
원본,19617
중복 통합 후 (food_clean),18998
1차 추천 대상 (food_menu),8678
통합 매핑 (dedup_map),619


In [27]:
clean["메뉴그룹"].value_counts()

메뉴그룹
식사     8678
음료     5768
디저트    3832
반찬      704
기타       16
Name: count, dtype: int64

### 결측치 비교

In [28]:
shared_cols = [c for c in clean.columns if c in raw.columns]
pd.DataFrame({
    "원본 결측(%)": (raw[shared_cols].isna().mean() * 100).round(1),
    "정제 후 결측(%)": (clean[shared_cols].isna().mean() * 100).round(1),
    "식사 대상 결측(%)": (menu[shared_cols].isna().mean() * 100).round(1),
})

,원본 결측(%),정제 후 결측(%),식사 대상 결측(%)
식품코드,0.0,0.0,0.0
식품명,0.0,0.0,0.0
식품대분류명,0.0,0.0,0.0
대표식품명,0.0,0.0,0.0
식품중분류명,0.0,85.9,79.5
식품기원명,0.0,0.0,0.0
업체명,0.0,18.6,31.1
출처명,0.0,0.0,0.0
데이터생성방법명,0.0,0.0,0.0
영양성분함량기준량,0.0,0.0,0.0


In [29]:
# 새로 추가된 컬럼의 결측 (온도, 사이즈, 접두어는 해당 없으면 결측이 정상)
new_cols = ["메뉴명", "이름접두어", "온도", "사이즈", "중량값", "중량단위", "메뉴그룹", "프랜차이즈여부"]
(clean[new_cols].isna().mean() * 100).round(1).rename("정제 후 결측(%)")

메뉴명         0.0
이름접두어      18.6
온도         82.4
사이즈        64.1
중량값         0.3
중량단위        0.3
메뉴그룹        0.0
프랜차이즈여부     0.0
Name: 정제 후 결측(%), dtype: float64

### 중복 비교

In [30]:
pd.DataFrame({
    "원본": {
        "완전 중복": raw.duplicated().sum(),
        "식품명 중복": raw.duplicated("식품명").sum(),
        "식품명 + 업체명 중복": raw.duplicated(["식품명", "업체명"]).sum(),
    },
    "정제 후": {
        "완전 중복": clean.duplicated().sum(),
        "식품명 중복": clean.duplicated("식품명").sum(),
        "식품명 + 업체명 중복": clean.duplicated(["식품명", "업체명"]).sum(),
    },
})

,원본,정제 후
완전 중복,0,0
식품명 중복,3970,3351
식품명 + 업체명 중복,2236,1617


정제 후에도 식품명 중복이 남아 있는 것은 의도한 결과다. 같은 메뉴명이라도 업체나 영양성분이 다르면 별개 메뉴로 취급한다. 남은 `식품명 + 업체명` 중복은 사이즈나 온도만 다른 프랜차이즈 메뉴다.

In [31]:
dup_name_company = clean[clean.duplicated(["식품명", "업체명"], keep=False)]
dup_name_company[["식품명", "업체명", "사이즈", "온도", "중량값", "중량단위", "에너지(kcal)"]].head(6)

,식품명,업체명,사이즈,온도,중량값,중량단위,에너지(kcal)
3,흑미밥_찹쌀,NaN,NaN,NaN,190.0,ml,118
4,흑미밥_찹쌀,NaN,NaN,NaN,270.0,ml,121
5,흑미밥,NaN,NaN,NaN,390.0,ml,118
6,흑미밥,NaN,NaN,NaN,310.0,ml,118
7,흑미밥,NaN,NaN,NaN,250.0,ml,120
8,흑미밥,NaN,NaN,NaN,300.0,ml,120


### 대표 샘플 비교

In [32]:
compare_cols = ["식품코드", "식품명", "메뉴명", "이름접두어", "온도", "사이즈", "메뉴그룹", "업체명", "중량값", "중량단위"]
sample_codes = ["D504-212000000-0001", raw.loc[raw["식품명"].str.contains("아메리카노 아이스"), "식품코드"].iloc[0], raw.loc[raw["식품명"] == "된장국_근대", "식품코드"].iloc[0]]

print("원본")
display(raw.loc[raw["식품코드"].isin(sample_codes), ["식품코드", "식품명", "식품대분류명", "업체명", "식품중량"]])
print("정제 후")
display(clean.loc[clean["식품코드"].isin(sample_codes), compare_cols])

원본


,식품코드,식품명,식품대분류명,업체명,식품중량
0,D504-212000000-0001,흰죽,죽 및 스프류,해당없음,291.90ml
7079,D220-748080000-0117,커피_화이트 아메리카노 아이스(ICED) (Venti),음료 및 차류,커피에반하다,720ml
17208,D605-216050000-0001,된장국_근대,국 및 탕류,해당없음,200ml


정제 후


,식품코드,식품명,메뉴명,이름접두어,온도,사이즈,메뉴그룹,업체명,중량값,중량단위
0,D504-212000000-0001,흰죽,흰죽,NaN,NaN,NaN,식사,NaN,291.9,ml
7001,D220-748080000-0117,커피_화이트 아메리카노 아이스(ICED) (Venti),화이트 아메리카노,커피,ICED,Venti,음료,커피에반하다,720.0,ml
16759,D605-216050000-0001,된장국_근대,된장국 근대,NaN,NaN,NaN,식사,NaN,200.0,ml


In [33]:
menu.sample(10, random_state=3)[["식품코드", "메뉴명", "식품대분류명", "프랜차이즈여부", "업체명", "에너지(kcal)", "나트륨(mg)"]]

,식품코드,메뉴명,식품대분류명,프랜차이즈여부,업체명,에너지(kcal),나트륨(mg)
7236,D303-148440000-0001,라면 짜장라면,면 및 만두류,False,NaN,177,306.0
3206,D202-120000000-2699,블랙타이거 슈림프 피자 슈퍼시드 화이버 오리지널,빵 및 과자류,True,도미노피자,243,447.0
4384,D202-120000000-4116,닭발피자피자,빵 및 과자류,True,봉수아피자,156,226.0
1735,D202-120000000-4688,치킨 바베큐,빵 및 과자류,True,파파존스,228,573.0
4096,D202-120000000-0447,로맨틱 하와이안 피자 씬도우,빵 및 과자류,True,피자파는집,109,225.0
2836,D202-120000000-4410,슈퍼슈프림고구마크러스트,빵 및 과자류,True,지정환피자,243,366.0
1395,D202-120180000-0107,통마늘 불고기 피자밀도우,빵 및 과자류,True,피자파는집,70,131.0
1704,D202-120000000-1350,치킨바베큐 피자,빵 및 과자류,True,파파존스,251,590.0
5556,D701-078000000-0001,유부 초밥,밥류,False,NaN,199,134.0
812,D202-120000000-3742,프리미엄 직화불고기 피자 오리지널,빵 및 과자류,True,파파존스,272,578.0


## 정리

### 결정한 기준과 근거

- 1차 추천 대상은 메뉴그룹 `식사`다. 빵 및 과자류 중 피자, 버거, 햄버거, 샌드위치, 핫도그, 토스트는 한 끼 식사로 먹는 메뉴라 식사에 포함했다. 반찬은 단독 추천 메뉴로 부적절해 제외했다. 디저트, 음료, 기타는 라벨만 남겼다.
- 식품명은 원본을 보존하고 `메뉴명`을 추가했다. 프랜차이즈 접두어는 `이름접두어`, HOT/ICED는 `온도`, 사이즈 표기는 `사이즈`로 분리했다. 수량 표기(조각, N개입)는 메뉴 정보라 이름에 남겼다.
- 중복은 식품명, 업체명, 기준량, 중량, 영양성분 25개가 모두 같을 때만 통합했다. 결과적으로 급식 반복 데이터만 통합되었고 프랜차이즈 메뉴는 하나도 제거되지 않았다. 통합된 행은 `dedup_map.csv`로 추적한다.
- 영양성분 결측은 채우지 않았다. `해당없음`은 업체명과 중분류명에서 결측으로 바꾸고 소분류·세분류는 컬럼을 제외했다.
- 식품중량은 숫자와 단위로만 분리했다. 1인분 여부와 g/ml 환산은 근거가 없어 하지 않았다.